### Setup the environment (e.g. on Colab), skip if genexp already installed

Install Python 3.10 (on Colab)

In [ ]:
# Source - https://stackoverflow.com/a/79612856
# Posted by naveedk
# Retrieved 2026-06-08, License - CC BY-SA 4.0

!apt-get update -y
!apt-get install python3.10 python3.10-distutils

Select Python 3.10: **Colab will prompt you to choose the python version, make sure you select python 3.10**

In [ ]:
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!update-alternatives --config python3

!apt-get install python3-pip
!python3 -m pip install --upgrade pip --user

**Important**: once the cells above have run you must restart the runtime in Colab using the Runtime tab on top (Runtime -> Restart session) before continuing

Run this cell to install genexp into your environment if you haven't already done so (e.g. on Colab)

In [ ]:
!python -m pip install torch==2.3.* dgl==2.4 --find-links https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
!python -m pip install git+https://github.com/riccardodesanti/flow-density-control

In [ ]:
from matplotlib import pyplot as plt
import torch

import diffusiongym
from omegaconf import OmegaConf
from genexp.trainers.fdc import FDCTrainer

import torch.distributions as dist

### Setup the base model

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

env = diffusiongym.make(
    base_model="1d/trimodal_gmm",
    reward="1d/sigmoidal",
    discretization_steps=50,
    device=device,
)

# True GMM density for reference
p_true = dist.MixtureSameFamily(
    dist.Categorical(torch.tensor([0.8, 0.1, 0.1])),
    dist.Normal(torch.tensor([0.0, 1.0, -1.0]), torch.tensor([0.2, 0.1, 0.1])),
)

with torch.no_grad():
    samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()

x_range = torch.linspace(-3, 4, 500)
true_density = p_true.log_prob(x_range).exp().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    samples.numpy(),
    bins=80,
    density=True,
    alpha=0.6,
    label="model samples",
    color="steelblue",
)
ax.plot(x_range.numpy(), true_density, "r-", lw=2, label="true GMM density")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Trimodal GMM base model: large mode at 0, smaller modes at ±1")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
base_samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()

plt.hist(
    base_samples.numpy(),
    bins=50,
    density=True,
    alpha=0.5,
    label="pre-trained model density",
)
plt.legend()
plt.show()

### Setup and run FlowExpansion

In [ ]:
"""
Recall from before the env was set up as:

env = diffusiongym.make(
    base_model="1d/trimodal_gmm",
    reward="1d/sigmoidal",
    discretization_steps=50,
    device=device,
)

"""

config = OmegaConf.load("configs/example_fdc.yaml")
fdc_trainer = FDCTrainer(config, env, device=device)

# Main Flow Expansion loop
fdc_trainer.fit(config.num_md_iterations, pbar=True)

In [ ]:
# Visualizing fine-tuned density (takes a few minutes to sample enough points)

env.base_model = fdc_trainer.fine_model
fine_samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()
env.base_model = fdc_trainer.base_base_model

plt.hist(
    base_samples.numpy(),
    bins=50,
    density=True,
    alpha=0.5,
    label="pre-trained model density",
    color="orange",
)
plt.hist(
    fine_samples.numpy(),
    bins=50,
    density=True,
    alpha=0.5,
    label="fine-tuned model density",
    color="green",
)
plt.legend()
plt.show()